# 📊 SOS + IL — Pipeline Completo de Métricas
### Strength of Schedule · Z-Score Ajustado · Custo/Valor do Gol · Índice de Legitimidade

---

## O que este notebook faz (em ordem)

| Etapa | Métricas geradas |
|---|---|
| **1 — SOS** | `sos_home/away_raw/norm/n` — força da tabela via ELO ponderado |
| **2 — Z-Score** | `z_gols/xg/vit_home/away` + `_adj` — desvio da média ajustado pelo SOS |
| **3 — Custo/Valor do Gol** | `custo/valor_gol_casa/fora` — lógica Adriano Duarte com gols ajustados por xG |
| **4 — Luck Factor** | `luck_home/away` — ΔxG (gols − xG esperado) |
| **5 — Delta** | `delta_bc/bf` — divergência modelo vs mercado |
| **6 — IL** | `IL_casa/fora` + `classificacao_IL` — índice 0-100 de legitimidade |

---

## Equações centrais

```
SOS_raw      = Σ(ELO_adv_i × decay^i) / Σ(decay^i)      [últimos N jogos]
SOS_norm     = SOS_raw / ELO_médio_liga                    [1.0 = média da liga]
Z_adj        = Z_bruto / SOS_norm                          [tabela fácil amplifica]

confiança    = min(N_jogos / N_REF, 1.0)
alpha_ef     = ALPHA × confiança
Gols_adj     = alpha_ef × Gols_N10 + (1 - alpha_ef) × xG_N10

Custo_Casa   = Gols_adj_casa / (1/odd_casa)               [maior = melhor]
Valor_Casa   = Gols_adj_casa × (1/odd_fora)               [maior = melhor]
Luck         = Gols_N10 − xG_N10                           [próximo 0 = saudável]
Delta        = prob_modelo − prob_mercado                   [>0 = modelo vê valor]

IL           = 0.25×EV + 0.20×SOS + 0.20×Z + 0.20×Custo + 0.15×Luck
```

## Classificação final

| Classificação | Condição |
|---|---|
| `VALOR_REAL_CASA/FORA` | IL > 0.65 + EV > 3% |
| `FALSO_FAVORITO_CASA/FORA` | prob_mercado > 55% + IL < 35% |
| `POTENCIAL_CASA/FORA` | Delta > 8% + IL > 50% |
| `NEUTRO` | Demais casos |


In [22]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 0 — Imports e Constantes
# Único ponto de configuração de todo o pipeline
# ═══════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from IPython.display import display

# ── Caminhos ──────────────────────────────────────────────
PATH_RODADA  = 'rodada_atual_v3.csv'      # entrada: rodada atual sem SOS/IL
PATH_BASE    = 'base_historica_v3.csv'    # entrada: base histórica com ELO
PATH_OUTPUT  = 'rodada_atual_sos_v3.csv'  # saída: CSV completo com todas as métricas

# ── Parâmetros SOS ────────────────────────────────────────
N_JOGOS = 10      # últimos N jogos para calcular SOS
DECAY   = 0.85    # decay exponencial: jogo[0]=1.0, jogo[1]=0.85, jogo[2]=0.72...

# ── Parâmetros RTM — μ/σ calibrados na base histórica (n≈5k jogos) ──
RTM_PARAMS = {
    'gols_home': {'mu': 1.5336, 'sigma': 0.6711, 'col': 'gols_marc_home'},
    'gols_away': {'mu': 1.2656, 'sigma': 0.5790, 'col': 'gols_marc_away'},
    'xg_home':   {'mu': 1.5592, 'sigma': 0.3601, 'col': 'xg_home_n10'},
    'xg_away':   {'mu': 1.3208, 'sigma': 0.3242, 'col': 'xg_away_n10'},
    'vit_home':  {'mu': 0.5002, 'sigma': 0.2456, 'col': 'pct_vit_home'},
    'vit_away':  {'mu': 0.3916, 'sigma': 0.2172, 'col': 'pct_vit_away'},
}

# ── Parâmetros IL ─────────────────────────────────────────
ALPHA_SHRINKAGE = 0.60   # peso gols reais vs xG âncora (N=10: alpha_ef=0.60, N=5: 0.30)
N_REF           = 10     # N de referência para confiança amostral
DELTA_THRESHOLD = 0.08   # divergência mínima modelo vs mercado para classificar Potencial

PESOS_IL = {
    'ev':    0.25,   # EV positivo    — mercado subestima
    'sos':   0.20,   # SOS alto       — forma veio de tabela difícil
    'z':     0.20,   # Z sustentável  — não está em pico atípico
    'custo': 0.20,   # Custo do Gol   — eficiência real contra expectativa
    'luck':  0.15,   # Luck baixo     — não depende de sorte na conversão
}

# ── Limiares de classificação IL ─────────────────────────
IL_VALOR_MIN   = 0.65   # IL acima = candidato a Valor Real
IL_FALSO_MAX   = 0.35   # IL abaixo + mercado favorece = Falso Favorito
PROB_MERC_MIN  = 0.55   # prob mercado acima = time considerado favorito
EV_MIN_IL      = 0.03   # EV mínimo para classificar Valor Real

print('✅ Constantes configuradas')
print(f'   Entrada : {PATH_RODADA} + {PATH_BASE}')
print(f'   Saída   : {PATH_OUTPUT}')
print(f'   SOS     : N={N_JOGOS} | decay={DECAY}')
print(f'   IL      : alpha={ALPHA_SHRINKAGE} | delta_th={DELTA_THRESHOLD}')
print(f'   Pesos IL: {PESOS_IL}')

✅ Constantes configuradas
   Entrada : rodada_atual_v3.csv + base_historica_v3.csv
   Saída   : rodada_atual_sos_v3.csv
   SOS     : N=10 | decay=0.85
   IL      : alpha=0.6 | delta_th=0.08
   Pesos IL: {'ev': 0.25, 'sos': 0.2, 'z': 0.2, 'custo': 0.2, 'luck': 0.15}


In [23]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 1 — Carregar CSVs
# ═══════════════════════════════════════════════════════════
ra = pd.read_csv(PATH_RODADA)
bh = pd.read_csv(PATH_BASE)

print(f'rodada_atual  : {ra.shape[0]} jogos | {ra.shape[1]} colunas')
print(f'base_historica: {bh.shape[0]} jogos | {bh.shape[1]} colunas')
print(f'Ligas rodada  : {ra["league"].nunique()} | Ligas base: {bh["league"].nunique()}')
print()

# Cobertura de IDs entre rodada e base
ids_ra = set(ra['home_id'].tolist() + ra['away_id'].tolist())
ids_bh = set(bh['home_id'].dropna().tolist()) | set(bh['away_id'].dropna().tolist())
cobertura = len(ids_ra & ids_bh) / len(ids_ra) * 100
print(f'Cobertura de times por ID: {cobertura:.0f}%')

# Avisar se colunas SOS/IL já existem (serão recalculadas)
cols_existentes = [c for c in ra.columns if 'sos' in c.lower() or c.startswith('z_')
                   or c.startswith('IL') or 'custo_gol' in c or 'luck_' in c]
if cols_existentes:
    print(f'⚠️  {len(cols_existentes)} colunas existentes serão recalculadas: {cols_existentes[:5]}...')
else:
    print('ℹ️  Calculando do zero')

display(ra[['league','home_team','away_team','elo_home','elo_away',
            'gols_marc_home','xg_home_n10','n10_home']].head(5))

rodada_atual  : 28 jogos | 62 colunas
base_historica: 5788 jogos | 78 colunas
Ligas rodada  : 5 | Ligas base: 28

Cobertura de times por ID: 100%
ℹ️  Calculando do zero


,league,home_team,away_team,elo_home,elo_away,gols_marc_home,xg_home_n10,n10_home
0,Espanha_2 Segunda Division,Albacete Balompié,CD Castellón,1519.939168,1554.369345,1.2,1.411,10
1,Espanha_2 Segunda Division,Almería,Real Sociedad II,1542.678195,1562.127944,1.8,1.516,10
2,Espanha_2 Segunda Division,Real Valladolid,Burgos CF,1432.820422,1544.394644,1.2,1.307,10
3,Espanha_2 Segunda Division,Ceuta,Cádiz,1490.933274,1468.075876,1.3,1.339,10
4,Espanha_2 Segunda Division,Málaga CF,Leganés,1538.323401,1456.536306,2.2,1.715,10


In [24]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 2 — Funções (SOS · Z-Score · IL)
# ═══════════════════════════════════════════════════════════

# ── SOS ──────────────────────────────────────────────────
def calc_sos(team_id, liga, bh_df, elo_liga_map, n=N_JOGOS, decay=DECAY):
    """
    SOS_raw  = média ponderada do ELO dos adversários (decay exponencial)
    SOS_norm = SOS_raw / ELO_médio_liga  →  1.0 = média da liga
    """
    j_casa = bh_df[bh_df['home_id'] == team_id][['date_unix','elo_away']].copy()
    j_casa.columns = ['date_unix','opp_elo']
    j_fora = bh_df[bh_df['away_id'] == team_id][['date_unix','elo_home']].copy()
    j_fora.columns = ['date_unix','opp_elo']

    todos = (
        pd.concat([j_casa, j_fora])
        .dropna(subset=['opp_elo'])
        .query('opp_elo > 1000')
        .sort_values('date_unix', ascending=False)
        .head(n)
        .reset_index(drop=True)
    )
    if len(todos) == 0:
        return None, None, 0

    pesos    = np.array([decay**i for i in range(len(todos))])
    sos_raw  = float(np.average(todos['opp_elo'], weights=pesos))
    elo_med  = elo_liga_map.get(liga, 1500.0)
    sos_norm = sos_raw / elo_med if elo_med > 0 else 1.0
    return round(sos_raw, 2), round(sos_norm, 4), len(todos)


# ── Z-Score ──────────────────────────────────────────────
def z_score(val, mu, sigma):
    """Z = (x − μ) / σ"""
    return round((val - mu) / sigma, 4) if sigma > 0 else 0.0

def z_ajustado(z_bruto, sos_norm):
    """Z_adj = Z_bruto / SOS_norm  →  tabela fácil amplifica o alerta"""
    sos = sos_norm if sos_norm and sos_norm > 0 else 1.0
    return round(z_bruto / sos, 4)


# ── IL ───────────────────────────────────────────────────
def odd_para_prob(serie_odd):
    """prob = 1 / odd  (retorna NaN para odds inválidas)"""
    o = pd.to_numeric(serie_odd, errors='coerce')
    return np.where(o > 1.0, 1.0 / o, np.nan)

def calcular_gols_ajustados(gols, xg, n, alpha=ALPHA_SHRINKAGE, n_ref=N_REF):
    """
    Gols_adj = alpha_ef × Gols_N10 + (1 − alpha_ef) × xG_N10
    alpha_ef = alpha × min(N/N_ref, 1.0)  →  N pequeno ancora mais no xG
    """
    confianca = np.minimum(n / n_ref, 1.0)
    alpha_ef  = alpha * confianca
    return alpha_ef * gols + (1.0 - alpha_ef) * xg

def normalizar_01(serie, q_low=0.05, q_high=0.95):
    """Normaliza para [0,1] usando percentis — robusto a outliers"""
    lo = serie.quantile(q_low)
    hi = serie.quantile(q_high)
    if hi == lo:
        return pd.Series(0.5, index=serie.index)
    return ((serie - lo) / (hi - lo)).clip(0, 1)

def classificar_jogo(row):
    """
    Hierarquia: FALSO_FAVORITO > VALOR_REAL > POTENCIAL > NEUTRO
    """
    il_h  = row['IL_casa']
    il_a  = row['IL_fora']
    pm_h  = row.get('prob_mercado_casa') or 0
    pm_a  = row.get('prob_mercado_fora') or 0
    ev_h  = row['ev_bc']  if pd.notna(row.get('ev_bc'))  else 0
    ev_a  = row['ev_bf']  if pd.notna(row.get('ev_bf'))  else 0
    dlt_h = row.get('delta_bc') or 0
    dlt_a = row.get('delta_bf') or 0

    if pm_h > PROB_MERC_MIN and il_h < IL_FALSO_MAX:
        return 'FALSO_FAVORITO_CASA'
    if pm_a > PROB_MERC_MIN and il_a < IL_FALSO_MAX:
        return 'FALSO_FAVORITO_FORA'
    if il_h > IL_VALOR_MIN and ev_h > EV_MIN_IL:
        return 'VALOR_REAL_CASA'
    if il_a > IL_VALOR_MIN and ev_a > EV_MIN_IL:
        return 'VALOR_REAL_FORA'
    if dlt_h > DELTA_THRESHOLD and il_h > 0.50:
        return 'POTENCIAL_CASA'
    if dlt_a > DELTA_THRESHOLD and il_a > 0.50:
        return 'POTENCIAL_FORA'
    return 'NEUTRO'


print('✅ Funções definidas: calc_sos | z_score | z_ajustado | odd_para_prob')
print('                     calcular_gols_ajustados | normalizar_01 | classificar_jogo')

✅ Funções definidas: calc_sos | z_score | z_ajustado | odd_para_prob
                     calcular_gols_ajustados | normalizar_01 | classificar_jogo


In [25]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 3 — ELO médio por liga + SOS
# ═══════════════════════════════════════════════════════════
import time

# ELO médio por liga (base de normalização do SOS)
elo_liga = {}
for liga in bh['league'].unique():
    sub  = bh[bh['league'] == liga]
    elos = pd.concat([sub['elo_home'], sub['elo_away']]).dropna()
    er   = elos[elos != 1500.0]  # excluir ELO padrão inicial
    elo_liga[liga] = float(er.mean()) if len(er) > 10 else 1500.0

print(f'ELO médio calculado para {len(elo_liga)} ligas')
print()

# Calcular SOS para todos os times da rodada
print(f'Calculando SOS para {len(ra)} jogos ({len(ra)*2} times)...')
sh_r, sh_n, sh_c = [], [], []
sa_r, sa_n, sa_c = [], [], []

for _, row in ra.iterrows():
    r, n, c = calc_sos(row['home_id'], row['league'], bh, elo_liga)
    sh_r.append(r); sh_n.append(n); sh_c.append(c)
    r, n, c = calc_sos(row['away_id'], row['league'], bh, elo_liga)
    sa_r.append(r); sa_n.append(n); sa_c.append(c)

for col in ['sos_home_raw','sos_home_norm','sos_home_n',
            'sos_away_raw','sos_away_norm','sos_away_n']:
    if col in ra.columns: ra.drop(columns=[col], inplace=True)

ra['sos_home_raw']  = sh_r
ra['sos_home_norm'] = sh_n
ra['sos_home_n']    = sh_c
ra['sos_away_raw']  = sa_r
ra['sos_away_norm'] = sa_n
ra['sos_away_n']    = sa_c

print(f'✅ SOS calculado')
print(f'   home: min={ra["sos_home_norm"].min():.4f} max={ra["sos_home_norm"].max():.4f} μ={ra["sos_home_norm"].mean():.4f}')
print(f'   away: min={ra["sos_away_norm"].min():.4f} max={ra["sos_away_norm"].max():.4f} μ={ra["sos_away_norm"].mean():.4f}')

display(ra[['league','home_team','away_team',
            'sos_home_norm','sos_home_n',
            'sos_away_norm','sos_away_n']].head(6))

ELO médio calculado para 28 ligas

Calculando SOS para 28 jogos (56 times)...
✅ SOS calculado
   home: min=0.9745 max=1.0123 μ=0.9983
   away: min=0.9717 max=1.0144 μ=1.0007


,league,home_team,away_team,sos_home_norm,sos_home_n,sos_away_norm,sos_away_n
0,Espanha_2 Segunda Division,Albacete Balompié,CD Castellón,1.0119,10,0.9934,10
1,Espanha_2 Segunda Division,Almería,Real Sociedad II,0.9745,10,1.0144,10
2,Espanha_2 Segunda Division,Real Valladolid,Burgos CF,0.9933,10,0.9926,10
3,Espanha_2 Segunda Division,Ceuta,Cádiz,0.9936,10,0.9892,10
4,Espanha_2 Segunda Division,Málaga CF,Leganés,0.9818,10,1.0070,10
5,Espanha_2 Segunda Division,Deportivo La Coruña,Córdoba,1.0098,10,1.0064,10


In [26]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 4 — Z-Scores brutos + ajustados por SOS
# ═══════════════════════════════════════════════════════════

for key, p in RTM_PARAMS.items():
    col_z    = f'z_{key}'
    col_zadj = f'z_{key}_adj'

    if p['col'] not in ra.columns:
        print(f'⚠️  Coluna {p["col"]} não encontrada — pulando {key}')
        continue

    for c in [col_z, col_zadj]:
        if c in ra.columns: ra.drop(columns=[c], inplace=True)

    # Z bruto
    ra[col_z] = ra[p['col']].apply(
        lambda x: z_score(float(x) if pd.notna(x) else p['mu'], p['mu'], p['sigma'])
    )

    # Z ajustado: home → sos_home_norm | away → sos_away_norm
    sos_col = 'sos_home_norm' if 'home' in key else 'sos_away_norm'
    ra[col_zadj] = ra.apply(lambda r: z_ajustado(r[col_z], r[sos_col]), axis=1)

z_cols = [c for c in ra.columns if c.startswith('z_')]
print(f'✅ Z-Scores calculados: {z_cols}')
print()
print('Amostra Z bruto vs Z_adj:')
display(ra[['home_team','away_team',
            'z_gols_home','sos_home_norm','z_gols_home_adj',
            'z_gols_away','sos_away_norm','z_gols_away_adj']].head(6))

✅ Z-Scores calculados: ['z_gols_home', 'z_gols_home_adj', 'z_gols_away', 'z_gols_away_adj', 'z_xg_home', 'z_xg_home_adj', 'z_xg_away', 'z_xg_away_adj', 'z_vit_home', 'z_vit_home_adj', 'z_vit_away', 'z_vit_away_adj']

Amostra Z bruto vs Z_adj:


,home_team,away_team,z_gols_home,sos_home_norm,z_gols_home_adj,z_gols_away,sos_away_norm,z_gols_away_adj
0,Albacete Balompié,CD Castellón,-0.4971,1.0119,-0.4913,0.5775,0.9934,0.5813
1,Almería,Real Sociedad II,0.3970,0.9745,0.4074,1.6138,1.0144,1.5909
2,Real Valladolid,Burgos CF,-0.4971,0.9933,-0.5005,-0.2860,0.9926,-0.2881
3,Ceuta,Cádiz,-0.3481,0.9936,-0.3503,-0.2860,0.9892,-0.2891
4,Málaga CF,Leganés,0.9930,0.9818,1.0114,-0.1133,1.0070,-0.1125
5,Deportivo La Coruña,Córdoba,-0.6461,1.0098,-0.6398,0.4048,1.0064,0.4022


In [27]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 5 — Custo/Valor do Gol · Luck Factor · Delta
# ═══════════════════════════════════════════════════════════

# Odds: preferir atual, fallback para D-2
odd_home = ra['odd_home_atual'].where(
    pd.to_numeric(ra['odd_home_atual'], errors='coerce') > 1.0, ra['odd_home_d2'])
odd_away = ra['odd_away_atual'].where(
    pd.to_numeric(ra['odd_away_atual'], errors='coerce') > 1.0, ra['odd_away_d2'])

prob_casa = odd_para_prob(odd_home)
prob_fora = odd_para_prob(odd_away)

ra['prob_mercado_casa'] = prob_casa.round(4)
ra['prob_mercado_fora'] = prob_fora.round(4)

# Gols ajustados (xG âncora + peso amostral)
gols_h_adj = calcular_gols_ajustados(ra['gols_marc_home'], ra['xg_home_n10'], ra['n10_home'])
gols_a_adj = calcular_gols_ajustados(ra['gols_marc_away'], ra['xg_away_n10'], ra['n10_away'])
ra['gols_home_adj'] = gols_h_adj.round(3)
ra['gols_away_adj'] = gols_a_adj.round(3)

# Custo do Gol  → Gols_adj / prob_própria   (maior = melhor)
# Valor do Gol  → Gols_adj × prob_adversário (maior = melhor)
ra['custo_gol_casa'] = (gols_h_adj / prob_casa).round(4)
ra['custo_gol_fora'] = (gols_a_adj / prob_fora).round(4)
ra['valor_gol_casa'] = (gols_h_adj * prob_fora).round(4)
ra['valor_gol_fora'] = (gols_a_adj * prob_casa).round(4)

# Luck Factor  →  ΔxG = Gols − xG  (próximo de 0 = saudável)
ra['luck_home'] = (ra['gols_marc_home'] - ra['xg_home_n10']).round(3)
ra['luck_away'] = (ra['gols_marc_away'] - ra['xg_away_n10']).round(3)

# Delta  →  prob_modelo − prob_mercado  (>0 = modelo vê mais valor)
ra['delta_bc'] = (ra['prob_bc'] - prob_casa).round(4)
ra['delta_bf'] = (ra['prob_bf'] - prob_fora).round(4)

print('✅ Custo/Valor do Gol, Luck Factor e Delta calculados')
display(ra[['home_team','away_team',
            'gols_marc_home','gols_home_adj','xg_home_n10',
            'custo_gol_casa','valor_gol_casa',
            'luck_home','delta_bc']].head(6))

✅ Custo/Valor do Gol, Luck Factor e Delta calculados


,home_team,away_team,gols_marc_home,gols_home_adj,xg_home_n10,custo_gol_casa,valor_gol_casa,luck_home,delta_bc
0,Albacete Balompié,CD Castellón,1.2,1.284,1.411,4.4183,0.6265,-0.211,0.0426
1,Almería,Real Sociedad II,1.8,1.686,1.516,2.6308,0.3123,0.284,-0.1879
2,Real Valladolid,Burgos CF,1.2,1.243,1.307,2.7963,0.3677,-0.107,-0.0845
3,Ceuta,Cádiz,1.3,1.316,1.339,2.8680,0.3836,-0.039,0.0268
4,Málaga CF,Leganés,2.2,2.006,1.715,4.1123,0.5378,0.485,0.0942
5,Deportivo La Coruña,Córdoba,1.1,1.207,1.368,NaN,NaN,-0.268,NaN


In [28]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 6 — Índice de Legitimidade (IL) + Classificação
# ═══════════════════════════════════════════════════════════

# Componentes casa (cada um normalizado em [0,1])
c1_h = normalizar_01(ra['ev_bc'].fillna(0))               # EV+
c2_h = normalizar_01(ra['sos_home_norm'])                  # SOS alto
c3_h = normalizar_01(-ra['z_gols_home_adj'].abs())         # Z próximo de 0
c4_h = normalizar_01(ra['custo_gol_casa'])                 # Custo alto
c5_h = normalizar_01(-ra['luck_home'].abs())               # Luck próximo de 0

ra['IL_casa'] = (
    PESOS_IL['ev']    * c1_h +
    PESOS_IL['sos']   * c2_h +
    PESOS_IL['z']     * c3_h +
    PESOS_IL['custo'] * c4_h +
    PESOS_IL['luck']  * c5_h
).round(4)

# Componentes fora
c1_a = normalizar_01(ra['ev_bf'].fillna(0))
c2_a = normalizar_01(ra['sos_away_norm'])
c3_a = normalizar_01(-ra['z_gols_away_adj'].abs())
c4_a = normalizar_01(ra['custo_gol_fora'])
c5_a = normalizar_01(-ra['luck_away'].abs())

ra['IL_fora'] = (
    PESOS_IL['ev']    * c1_a +
    PESOS_IL['sos']   * c2_a +
    PESOS_IL['z']     * c3_a +
    PESOS_IL['custo'] * c4_a +
    PESOS_IL['luck']  * c5_a
).round(4)

# Classificação
ra['classificacao_IL'] = ra.apply(classificar_jogo, axis=1)

print('✅ IL calculado')
print(f'   IL_casa: min={ra["IL_casa"].min():.3f} max={ra["IL_casa"].max():.3f} μ={ra["IL_casa"].mean():.3f}')
print(f'   IL_fora: min={ra["IL_fora"].min():.3f} max={ra["IL_fora"].max():.3f} μ={ra["IL_fora"].mean():.3f}')
print()
print('Distribuição das classificações:')
print(ra['classificacao_IL'].value_counts())

✅ IL calculado
   IL_casa: min=0.250 max=0.912 μ=0.533
   IL_fora: min=0.220 max=0.809 μ=0.526

Distribuição das classificações:
classificacao_IL
NEUTRO                 18
VALOR_REAL_FORA         4
VALOR_REAL_CASA         3
FALSO_FAVORITO_CASA     1
POTENCIAL_CASA          1
FALSO_FAVORITO_FORA     1
Name: count, dtype: int64


In [29]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 7 — Análise por classificação
# ═══════════════════════════════════════════════════════════

cols_casa = ['league','home_team','away_team',
             'IL_casa','custo_gol_casa','valor_gol_casa',
             'luck_home','delta_bc','sos_home_norm','z_gols_home_adj',
             'ev_bc','prob_bc','prob_mercado_casa']
cols_fora = ['league','home_team','away_team',
             'IL_fora','custo_gol_fora','valor_gol_fora',
             'luck_away','delta_bf','sos_away_norm','z_gols_away_adj',
             'ev_bf','prob_bf','prob_mercado_fora']

for cl, cols in [
    ('VALOR_REAL_CASA',      cols_casa),
    ('VALOR_REAL_FORA',      cols_fora),
    ('FALSO_FAVORITO_CASA',  cols_casa),
    ('FALSO_FAVORITO_FORA',  cols_fora),
    ('POTENCIAL_CASA',       cols_casa),
    ('POTENCIAL_FORA',       cols_fora),
]:
    sub = ra[ra['classificacao_IL'] == cl]
    if len(sub) == 0: continue
    col_il = 'IL_casa' if 'CASA' in cl else 'IL_fora'
    print(f'\n=== {cl} ({len(sub)} jogos) ===')
    display(sub.sort_values(col_il, ascending='FALSO' in cl)[cols])


=== VALOR_REAL_CASA (3 jogos) ===


,league,home_team,away_team,IL_casa,custo_gol_casa,valor_gol_casa,luck_home,delta_bc,sos_home_norm,z_gols_home_adj,ev_bc,prob_bc,prob_mercado_casa
0,Espanha_2 Segunda Division,Albacete Balompié,CD Castellón,0.9124,4.4183,0.6265,-0.211,0.0426,1.0119,-0.4913,0.146476,0.333278,0.2907
6,Espanha_2 Segunda Division,SD Eibar,UD Las Palmas,0.7379,3.0488,0.3715,-0.111,0.0510,0.9981,-0.4980,0.125072,0.459213,0.4082
3,Espanha_2 Segunda Division,Ceuta,Cádiz,0.6853,2.8680,0.3836,-0.039,0.0268,0.9936,-0.3503,0.058525,0.485562,0.4587



=== VALOR_REAL_FORA (4 jogos) ===


,league,home_team,away_team,IL_fora,custo_gol_fora,valor_gol_fora,luck_away,delta_bf,sos_away_norm,z_gols_away_adj,ev_bf,prob_bf,prob_mercado_fora
19,Colombia_1 Liga BetPlay,América de Cali,Llaneros,0.8092,7.1880,0.7003,0.019,0.0636,0.9981,-0.2865,0.418738,0.215614,0.1520
21,Colombia_1 Liga BetPlay,Junior,Atlético Bucaramanga,0.7468,4.8370,0.5945,0.141,0.0930,1.0079,0.2303,0.334890,0.370803,0.2778
23,Colombia_1 Liga BetPlay,Fortaleza CEIF,Deportivo Pasto,0.7326,5.1903,0.6393,-0.022,0.1372,0.9993,0.4051,0.472007,0.427909,0.2907
7,Espanha_2 Segunda Division,Cultural y Deportiva Leonesa,FC Andorra,0.6542,4.5228,0.6282,-0.269,0.0637,1.0085,0.2301,0.191220,0.397073,0.3333



=== FALSO_FAVORITO_CASA (1 jogos) ===


,league,home_team,away_team,IL_casa,custo_gol_casa,valor_gol_casa,luck_home,delta_bc,sos_home_norm,z_gols_home_adj,ev_bc,prob_bc,prob_mercado_casa
1,Espanha_2 Segunda Division,Almería,Real Sociedad II,0.3489,2.6308,0.3123,0.284,-0.1879,0.9745,0.4074,-0.293166,0.453099,0.641



=== FALSO_FAVORITO_FORA (1 jogos) ===


,league,home_team,away_team,IL_fora,custo_gol_fora,valor_gol_fora,luck_away,delta_bf,sos_away_norm,z_gols_away_adj,ev_bf,prob_bf,prob_mercado_fora
13,Argentina_1 Liga Profesional,Aldosivi,Argentinos Juniors,0.2196,2.1512,0.2397,-1.208,-0.0433,1.0034,-1.0309,-0.076271,0.524846,0.5682



=== POTENCIAL_CASA (1 jogos) ===


,league,home_team,away_team,IL_casa,custo_gol_casa,valor_gol_casa,luck_home,delta_bc,sos_home_norm,z_gols_home_adj,ev_bc,prob_bc,prob_mercado_casa
4,Espanha_2 Segunda Division,Málaga CF,Leganés,0.5927,4.1123,0.5378,0.485,0.0942,0.9818,1.0114,0.193047,0.581974,0.4878


In [30]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 8 — Exportar CSV final
# ═══════════════════════════════════════════════════════════

ra.to_csv(PATH_OUTPUT, index=False)

todas_novas = [
    # SOS
    'sos_home_raw','sos_home_norm','sos_home_n',
    'sos_away_raw','sos_away_norm','sos_away_n',
    # Z-Scores
    'z_gols_home','z_gols_home_adj','z_gols_away','z_gols_away_adj',
    'z_xg_home','z_xg_home_adj','z_xg_away','z_xg_away_adj',
    'z_vit_home','z_vit_home_adj','z_vit_away','z_vit_away_adj',
    # IL
    'prob_mercado_casa','prob_mercado_fora',
    'gols_home_adj','gols_away_adj',
    'custo_gol_casa','custo_gol_fora',
    'valor_gol_casa','valor_gol_fora',
    'luck_home','luck_away',
    'delta_bc','delta_bf',
    'IL_casa','IL_fora','classificacao_IL',
]

print(f'✅ Exportado: {PATH_OUTPUT}')
print(f'   Shape final: {ra.shape}  (+{len(todas_novas)} colunas novas)')
print()
print('Colunas geradas neste pipeline:')
grupos = [
    ('SOS (6)',     [c for c in todas_novas if 'sos' in c]),
    ('Z-Score (12)',[c for c in todas_novas if c.startswith('z_')]),
    ('IL (15)',     [c for c in todas_novas if c not in
                    [c2 for c2 in todas_novas if 'sos' in c2 or c2.startswith('z_')]]),
]
for nome, cols in grupos:
    presentes = [c for c in cols if c in ra.columns]
    print(f'  {nome}: {presentes}')

print()
print('Distribuição final das classificações:')
print(ra['classificacao_IL'].value_counts())
print()
print('📌 Próximo passo: envie rodada_atual_sos_v3.csv para o Claude')
print('   → gerar index.html com SOS+RTM+IL embutido no painel.')
display(ra[['home_team','away_team','IL_casa','IL_fora','classificacao_IL']].head(8))

✅ Exportado: rodada_atual_sos_v3.csv
   Shape final: (28, 95)  (+33 colunas novas)

Colunas geradas neste pipeline:
  SOS (6): ['sos_home_raw', 'sos_home_norm', 'sos_home_n', 'sos_away_raw', 'sos_away_norm', 'sos_away_n']
  Z-Score (12): ['z_gols_home', 'z_gols_home_adj', 'z_gols_away', 'z_gols_away_adj', 'z_xg_home', 'z_xg_home_adj', 'z_xg_away', 'z_xg_away_adj', 'z_vit_home', 'z_vit_home_adj', 'z_vit_away', 'z_vit_away_adj']
  IL (15): ['prob_mercado_casa', 'prob_mercado_fora', 'gols_home_adj', 'gols_away_adj', 'custo_gol_casa', 'custo_gol_fora', 'valor_gol_casa', 'valor_gol_fora', 'luck_home', 'luck_away', 'delta_bc', 'delta_bf', 'IL_casa', 'IL_fora', 'classificacao_IL']

Distribuição final das classificações:
classificacao_IL
NEUTRO                 18
VALOR_REAL_FORA         4
VALOR_REAL_CASA         3
FALSO_FAVORITO_CASA     1
POTENCIAL_CASA          1
FALSO_FAVORITO_FORA     1
Name: count, dtype: int64

📌 Próximo passo: envie rodada_atual_sos_v3.csv para o Claude
   → gerar index

,home_team,away_team,IL_casa,IL_fora,classificacao_IL
0,Albacete Balompié,CD Castellón,0.9124,0.3232,VALOR_REAL_CASA
1,Almería,Real Sociedad II,0.3489,0.6500,FALSO_FAVORITO_CASA
2,Real Valladolid,Burgos CF,0.5086,0.5626,NEUTRO
3,Ceuta,Cádiz,0.6853,0.3656,VALOR_REAL_CASA
4,Málaga CF,Leganés,0.5927,0.5514,POTENCIAL_CASA
5,Deportivo La Coruña,Córdoba,NaN,NaN,NEUTRO
6,SD Eibar,UD Las Palmas,0.7379,0.5761,VALOR_REAL_CASA
7,Cultural y Deportiva Leonesa,FC Andorra,0.4278,0.6542,VALOR_REAL_FORA
